In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" 
      if torch.cuda.is_available() else "")

In [ ]:
def build_traffic_context(location_idx, locations, location_groups,
                           master, gnn_predictions, gnn_attention,
                           segment_mask):
    """
    Build structured natural language context for one location.
    This is the grounding layer — LLM never hallucinates numbers
    because everything is explicitly provided.
    """
    loc  = locations[location_idx]
    segs = location_groups[loc]
    pred = gnn_predictions[location_idx]   # (30,) km/h
    attn = gnn_attention[location_idx]     # (max_segments,)

    # ── Current state from most recent segment ────────────
    last_seg = segs[-1]
    subset   = master[master['source'] == last_seg].sort_values('frame_num')
    recent   = subset.tail(100)

    current_speed    = recent['avg_speed'].iloc[-1]
    current_vehicles = recent['vehicle_count'].iloc[-1]
    current_flow     = recent['total_flow'].iloc[-1]
    current_stopped  = recent['pct_stopped'].iloc[-1] * 100

    # ── Trend over last 20 frames ─────────────────────────
    trend_val  = recent['avg_speed'].tail(20).diff().mean()
    trend_word = ("worsening" if trend_val < -0.05 else
                  "improving" if trend_val > 0.05  else "stable")

    # ── Historical baseline ───────────────────────────────
    all_loc_data = master[master['source'].isin(segs)]
    hist_mean    = all_loc_data['avg_speed'].mean()
    hist_std     = all_loc_data['avg_speed'].std()
    hist_min     = all_loc_data['avg_speed'].min()
    hist_max     = all_loc_data['avg_speed'].max()

    vs_baseline = ((current_speed - hist_mean) / hist_mean * 100)
    severity    = ("critical"  if current_speed < hist_mean - 1.5*hist_std else
                   "severe"    if current_speed < hist_mean - hist_std     else
                   "moderate"  if current_speed < hist_mean                else
                   "light"     if current_speed < hist_mean + hist_std     else
                   "free flow")

    # ── Forecast summary ──────────────────────────────────
    pred_mean  = pred.mean()
    pred_min   = pred.min()
    pred_max   = pred.max()
    pred_trend = ("worsening" if pred[-1] < pred[0] - 0.3 else
                  "improving" if pred[-1] > pred[0] + 0.3 else "stable")
    pred_trajectory = [round(float(v), 2) for v in pred[:10]]

    # ── Dominant segment from attention ──────────────────
    real_attn   = attn[:len(segs)]
    dom_idx     = int(np.argmax(real_attn))
    dom_seg     = segs[dom_idx]
    dom_weight  = float(real_attn[dom_idx])

    # Build context string
    context = f"""
LOCATION: {loc} (Hohhot, Inner Mongolia, China)
RECORDING SEGMENTS ANALYZED: {len(segs)} ({', '.join(segs)})

CURRENT TRAFFIC STATE (most recent observation):
- Average speed: {current_speed:.2f} km/h
- Vehicle count: {int(current_vehicles)} vehicles in scene
- Total directional flow: {int(current_flow)} vehicles
- Percentage stopped: {current_stopped:.1f}%
- Short-term trend (last 20 frames): {trend_word}
- Congestion severity: {severity}
- vs. historical baseline: {vs_baseline:+.1f}% {'below' if vs_baseline < 0 else 'above'} normal

HISTORICAL CONTEXT FOR THIS LOCATION:
- Typical average speed: {hist_mean:.2f} km/h (±{hist_std:.2f})
- Observed range: {hist_min:.2f} – {hist_max:.2f} km/h
- Most informative recording session: {dom_seg} (attention weight: {dom_weight:.3f})
- This means: {dom_seg} best represents this location's typical traffic behavior

GNN FORECAST (next 30 frames):
- Predicted speed trajectory: {pred_trend}
- Predicted mean speed: {pred_mean:.2f} km/h
- Predicted range: {pred_min:.2f} – {pred_max:.2f} km/h
- First 10 predicted values (km/h): {pred_trajectory}
"""
    return context, severity, pred_trend


# Test the context builder on one location
ctx, sev, trend = build_traffic_context(
    0, locations, location_groups,
    master, gnn_predictions, gnn_attention, segment_mask
)
print(ctx)

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image
import torch
import numpy as np

# ── Load model ────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2-VL-7B-Instruct"

print("Loading Qwen2-VL-7B...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # half precision to save VRAM
    device_map="auto"            # automatically places layers on GPU
)
print("Model loaded")


def query_qwen_text_only(context, question=None):
    """
    Query Qwen2-VL with text context only (no image).
    Drop-in replacement for query_llm() from before.
    """
    user_content = f"""You are an expert traffic management AI for 
Hohhot, Inner Mongolia. Analyze this traffic data:

{context}

{"Question: " + question if question else 
 "Provide a complete traffic situation analysis."}"""

    messages = [{"role": "user", "content": user_content}]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True
        )

    generated = output_ids[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=True)[0]


def query_qwen_with_frame(context, image_path, question=None):
    """
    Query Qwen2-VL with CCTV frame + quantitative context.
    This is the multimodal capability that makes Qwen2-VL special.
    """
    image = Image.open(image_path).convert("RGB")

    user_content = [
        {"type": "image",  "image": image},
        {"type": "text",   "text": f"""You are an expert traffic 
management AI for Hohhot, Inner Mongolia. 

Look at this CCTV frame and analyze it together with the 
quantitative pipeline data below:

{context}

{"Question: " + question if question else 
 "Describe what you see in the frame and how it relates "
 "to the quantitative data. Identify causes of congestion "
 "visible in the image and provide recommendations."}"""}
    ]

    messages = [{"role": "user", "content": user_content}]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True
        )

    generated = output_ids[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=True)[0]


# ── Test text-only first ──────────────────────────────────
ctx, severity, pred_trend = build_traffic_context(
    0, locations, location_groups,
    master, gnn_predictions, gnn_attention, segment_mask
)

print("Testing Qwen2-VL text reasoning...")
response = query_qwen_text_only(ctx)
print(response)

In [ ]:
# If you get CUDA out of memory, use this instead
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)
# Reduces VRAM from ~16GB to ~5GB with minor quality loss

In [ ]:
# Extract a frame from your video at a specific timestamp
def extract_video_frame(video_path, frame_number, output_path):
    import cv2
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ret, frame = cap.read()
    if ret:
        cv2.imwrite(output_path, frame)
    cap.release()
    return output_path

# Then query with both vision and quantitative data
frame_path = extract_video_frame(
    '../data/hohhot_traffic.mp4',
    frame_number=12000,            # the anomaly frame we found earlier
    output_path='../data/frame_12000.jpg'
)

response = query_qwen_with_frame(
    context=ctx,
    image_path=frame_path,
    question="What do you see in this frame that explains "
             "the sudden drop in congestion score around "
             "frame 12000? Is this a real traffic clearance "
             "or a sensor/detection failure?"
)
print(response)